# SBND RWM True Bucket: Fetch Raw RWM Traces (Optional)

- Rebuilds the raw-trace store from IFBeam (device `E:NBMI12WV[]`) for every key in `inputs/traces_run{1,2}.csv`; needs the Fermilab network
- Only keys not in the store yet are queried; the default input is the stored copy on Fermilab (README)

## 1. Setup

- Set the path in the Setup cell: `STORE` = folders the traces are written to

In [ ]:
import os
import time
os.environ['TZ'] = 'UTC'
time.tzset()

import numpy as np
import pandas as pd

import sys
sys.path.insert(0, '../lib')
import paths
import ad_trace as at

# ===== Set this path =====
STORE = {1: '/path/to/run1_rwm_traces', 2: '/path/to/run2_rwm_traces'}   # stored raw RWM traces: folders of run_<DAQ run>.npz
paths.set_paths(store=STORE)

## 2. Fetch

- SBND-event keys (`event` ≥ 0): the record 35.5 ms before the event; time-sampled keys (`event` = −1): the first record after `ts_ns`
- `status` 0 = no IFBeam record; failed queries are not stored and are retried on the next run

In [ ]:
for run in (1, 2):
    lst = pd.read_csv(f'{paths.INPUTS}/traces_run{run}.csv')
    got = []
    for matched, fetch in ((True, at.fetch_trace), (False, at.fetch_trace_at)):
        sel = lst[(lst['event'] >= 0) == matched]
        if len(sel):
            got.append(at.ensure(paths.STORE[run], sel[['run', 'event', 'ts_ns']], fetch=fetch))
    n_key, n_rec = sum(len(d['run']) for d in got), sum(int(d['status'].sum()) for d in got)
    print(f"Run {run}: {n_key} / {len(lst)} keys stored, {n_rec} records (list: {int(lst['status'].sum())})")